# PyTorch Tutorial 46: Edge ML Fundamentals

**Author:** PyTorch Tutorial Series  
**Date:** 2026  
**Prerequisites:** Notebooks 00-05 (basic PyTorch)  
**Time:** ~1 hour

---

## What You'll Learn

1. **What is Edge ML** — and why it's one of the biggest trends in AI right now
2. **The Size Problem** — why most AI models are too big for small devices
3. **Edge-Friendly Models** — what makes a model suitable for phones, cameras, and sensors
4. **The Hardware Landscape** — from $5 microcontrollers to $2000 AI boards

---

## 1. What is Edge ML?

Most AI today runs in the **cloud** — your phone sends data to a massive server, the server does the thinking, and sends back the answer.

**Edge ML** flips this: the AI runs **directly on the device** — your phone, a security camera, a robot, or even a $5 microcontroller.

### Real Examples You've Already Used

| Application | What's happening on-device |
|-------------|---------------------------|
| Face unlock on your phone | A neural network recognizes your face in ~50ms |
| "Hey Siri" / "OK Google" | A tiny model listens for wake words 24/7 |
| Phone camera portrait mode | A model segments you from the background in real-time |
| Smart doorbell alerts | Object detection tells a person from a package |
| Predictive text keyboard | A small language model suggests your next word |

All of these run **locally** — no internet needed.

### Why Not Just Use the Cloud?

| Problem | Cloud | Edge |
|---------|-------|------|
| **Latency** | 100-500ms round trip | <10ms on-device |
| **Privacy** | Your data leaves your device | Data never leaves |
| **Cost** | Pay per API call ($$$) | Free after deployment |
| **Offline** | No internet = no AI | Works anywhere |
| **Bandwidth** | Sends raw data (images, audio) | Sends only results |

**The tradeoff?** Edge devices have limited compute and memory. A phone has ~4-8 GB RAM. A cloud GPU has 80 GB. So we need **smaller, faster models**.

## 2. The Size Problem

Let's see exactly how big typical AI models are — and why that's a problem for small devices.

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import time

print("PyTorch version:", torch.__version__)
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")

### How Big is a Model?

Every neural network is made of **parameters** (weights and biases). Each parameter is typically a 32-bit float = **4 bytes**.

So: `model size (MB) = number of parameters x 4 bytes / 1,000,000`

Let's write a simple function to check this:

In [ ]:
def get_model_size(model):
    """Calculate model size in MB and count parameters."""
    total_params = sum(p.numel() for p in model.parameters())
    size_mb = total_params * 4 / (1024 * 1024)  # 4 bytes per float32 param
    return total_params, size_mb


# Let's check some popular models
print(f"{'Model':<25} {'Parameters':>15} {'Size (MB)':>12}")
print("-" * 55)

model_configs = [
    ("MobileNetV3-Small", models.mobilenet_v3_small(weights=None)),
    ("MobileNetV3-Large", models.mobilenet_v3_large(weights=None)),
    ("ResNet-18", models.resnet18(weights=None)),
    ("ResNet-50", models.resnet50(weights=None)),
    ("EfficientNet-B0", models.efficientnet_b0(weights=None)),
    ("VGG-16", models.vgg16(weights=None)),
]

for name, model in model_configs:
    params, size = get_model_size(model)
    print(f"{name:<25} {params:>15,} {size:>10.1f} MB")

**Key takeaway:** MobileNetV3-Small is ~10 MB. VGG-16 is ~528 MB. That's a **50x difference!**

Now imagine your target device:

| Device | Available RAM for ML | Can it fit VGG-16? | Can it fit MobileNetV3? |
|--------|---------------------|--------------------|--------------------------|
| Cloud GPU (A100) | 80 GB | Yes, easily | Yes |
| Smartphone | 2-4 GB (shared) | Barely | Yes |
| Raspberry Pi 5 | 1-4 GB (shared) | Probably not | Yes |
| Microcontroller | 256 KB - 2 MB | No way | Still too big! |

This is why **model size matters** for edge deployment.

## 3. What Makes a Model "Edge-Friendly"?

Three things determine if a model can run well on a small device:

1. **Parameter count** — determines model size on disk/memory
2. **FLOPs** (Floating Point Operations) — determines how much compute is needed
3. **Peak memory** — determines RAM usage during inference

A model can be small (few params) but still slow (many FLOPs). Or big but fast. Let's measure all three.

In [ ]:
# Install torchinfo for easy model profiling
# !pip install torchinfo
from torchinfo import summary

# Profile MobileNetV3-Small — designed for edge
print("=" * 60)
print("MobileNetV3-Small (DESIGNED for phones)")
print("=" * 60)
mobile_model = models.mobilenet_v3_small(weights=None)
summary(mobile_model, input_size=(1, 3, 224, 224), verbose=0)

In [ ]:
# Profile ResNet-50 — designed for accuracy, not efficiency
print("=" * 60)
print("ResNet-50 (DESIGNED for accuracy)")
print("=" * 60)
resnet_model = models.resnet50(weights=None)
summary(resnet_model, input_size=(1, 3, 224, 224), verbose=0)

### Let's Measure Speed Too

Parameters and FLOPs are theory. Let's measure **actual inference time** on your CPU.

In [ ]:
def benchmark_model(model, input_shape=(1, 3, 224, 224), num_runs=50):
    """Measure average inference time on CPU."""
    model.eval()
    dummy_input = torch.randn(input_shape)

    # Warm-up (first run is always slow due to memory allocation)
    with torch.no_grad():
        for _ in range(5):
            model(dummy_input)

    # Actual benchmark
    times = []
    with torch.no_grad():
        for _ in range(num_runs):
            start = time.perf_counter()
            model(dummy_input)
            end = time.perf_counter()
            times.append((end - start) * 1000)  # Convert to ms

    avg_ms = sum(times) / len(times)
    return avg_ms


print(f"{'Model':<25} {'Params':>12} {'Size (MB)':>10} {'Latency (ms)':>14}")
print("-" * 65)

benchmark_models = [
    ("MobileNetV3-Small", models.mobilenet_v3_small(weights=None)),
    ("MobileNetV3-Large", models.mobilenet_v3_large(weights=None)),
    ("ResNet-18", models.resnet18(weights=None)),
    ("ResNet-50", models.resnet50(weights=None)),
    ("EfficientNet-B0", models.efficientnet_b0(weights=None)),
]

for name, model in benchmark_models:
    params, size = get_model_size(model)
    latency = benchmark_model(model)
    print(f"{name:<25} {params:>12,} {size:>8.1f} MB {latency:>12.1f} ms")

### The Design Tricks Behind MobileNet

Why is MobileNetV3 so much smaller and faster? It uses clever building blocks:

**1. Depthwise Separable Convolutions**

A regular convolution with a 3x3 kernel on 64 channels to 64 output channels uses:
- `3 x 3 x 64 x 64 = 36,864` parameters

A depthwise separable convolution splits this into two steps:
- Depthwise: `3 x 3 x 64 = 576` (one filter per channel)
- Pointwise: `1 x 1 x 64 x 64 = 4,096`
- Total: `4,672` — that's **8x fewer parameters!**

**2. Inverted Residuals**
- Expand channels, then depthwise conv, then squeeze channels
- Skip connection on the narrow (squeezed) representation

**3. Squeeze-and-Excite (SE) blocks**
- A tiny attention mechanism that learns "which channels matter"
- Adds <1% parameters but improves accuracy

Let's see the difference in practice:

In [ ]:
# Regular convolution vs Depthwise Separable
in_channels = 64
out_channels = 64

# Regular 3x3 convolution
regular_conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
regular_params = sum(p.numel() for p in regular_conv.parameters())

# Depthwise separable convolution (2 steps)
depthwise = nn.Conv2d(
    in_channels, in_channels, kernel_size=3, padding=1, groups=in_channels
)
pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1)
separable_params = sum(p.numel() for p in depthwise.parameters()) + \
                   sum(p.numel() for p in pointwise.parameters())

print(f"Regular Conv:     {regular_params:,} parameters")
print(f"Depthwise Sep.:   {separable_params:,} parameters")
print(f"Reduction:        {regular_params / separable_params:.1f}x fewer parameters!")

### Building a Tiny CNN for Edge

Let's build our own small model using these tricks and compare it with a naive approach.

In [ ]:
class NaiveCNN(nn.Module):
    """A simple CNN using regular convolutions."""

    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


class EdgeCNN(nn.Module):
    """A small CNN using depthwise separable convolutions."""

    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Initial regular conv (first layer is usually kept regular)
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            # Depthwise separable blocks
            nn.Conv2d(32, 32, 3, padding=1, groups=32), nn.ReLU(),
            nn.Conv2d(32, 64, 1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 64, 3, padding=1, groups=64), nn.ReLU(),
            nn.Conv2d(64, 128, 1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


naive = NaiveCNN()
edge = EdgeCNN()

naive_params, naive_size = get_model_size(naive)
edge_params, edge_size = get_model_size(edge)

naive_ms = benchmark_model(naive, input_shape=(1, 3, 32, 32))
edge_ms = benchmark_model(edge, input_shape=(1, 3, 32, 32))

print(f"{'Model':<20} {'Params':>10} {'Size (MB)':>10} {'Latency (ms)':>14}")
print("-" * 58)
print(f"{'NaiveCNN':<20} {naive_params:>10,} {naive_size:>8.2f} MB {naive_ms:>12.2f} ms")
print(f"{'EdgeCNN':<20} {edge_params:>10,} {edge_size:>8.2f} MB {edge_ms:>12.2f} ms")
print(f"\nEdgeCNN is {naive_params / edge_params:.1f}x smaller!")

## 4. The Hardware Zoo

Edge devices come in all shapes and sizes. Here's the landscape in 2026:

### High-Performance Edge ("AI at the Edge")

| Device | Compute (TOPS) | RAM | Price | Best For |
|--------|---------------|-----|-------|----------|
| NVIDIA Jetson AGX Orin | 275 | 32-64 GB | ~$2,000 | Autonomous vehicles, robots |
| NVIDIA Jetson Orin Nano Super | 67 | 8 GB | ~$249 | Gen AI at the edge, prototyping |
| Qualcomm Snapdragon 8 Gen 3 | 45 (NPU) | 8-16 GB | In phones | On-device LLMs, camera AI |
| Apple A17 Pro / M-series | ~35 (Neural Engine) | 6-128 GB | In devices | iOS/macOS ML apps |

### Mid-Range / Maker Boards

| Device | Compute (TOPS) | RAM | Price | Best For |
|--------|---------------|-----|-------|----------|
| Raspberry Pi 5 + Hailo-8 | 13-26 | 2-8 GB | ~$105 | Smart cameras, home automation |
| Google Coral (Edge TPU) | 4 | 1-4 GB | ~$60 | Object detection, classification |
| Rockchip RK3588 boards | 6 | 4-16 GB | ~$80 | Embedded Linux AI |

### Microcontrollers (TinyML)

| Device | Compute | RAM | Price | Best For |
|--------|---------|-----|-------|----------|
| ESP32-S3 | ~0.001 TOPS | 512 KB | ~$5 | Wake word, simple gestures |
| Arduino Nano 33 BLE | ~0.001 TOPS | 256 KB | ~$25 | Motion detection, sensor ML |
| STM32N6 (with NPU) | 0.6 TOPS | 4.2 MB | ~$15 | Vision on MCU |

### What are TOPS?

**TOPS** = Tera Operations Per Second (trillions of operations per second).

Think of it like horsepower for AI:
- 0.001 TOPS = bicycle (can run tiny keyword detection)
- 4 TOPS = compact car (can run object detection at 30fps)
- 45 TOPS = sports car (can run small language models)
- 275 TOPS = race car (can run large models in real-time)

### The NPU Revolution (2025-2026)

**NPU** = Neural Processing Unit — a chip designed specifically for AI inference.

In 2026, NPUs are now **standard** in:
- All flagship phones (Apple Neural Engine, Qualcomm Hexagon, Samsung Exynos)
- New laptops (Apple M-series, Intel Meteor Lake, Qualcomm Snapdragon X)
- Embedded SoCs (NXP, STMicroelectronics)

This means **most devices your users already own can run AI locally**.

> **Good news:** You don't need any of this hardware for this tutorial! We'll simulate everything on your regular computer. The techniques you learn here will work on any device.

## 5. Edge ML Decision Framework

When should you use edge vs cloud? Here's a simple decision tree:

In [ ]:
def should_use_edge(scenario):
    """
    Simple decision framework for edge vs cloud deployment.
    Returns a recommendation with reasoning.
    """
    reasons_for_edge = []
    reasons_for_cloud = []

    if scenario.get("latency_critical", False):
        reasons_for_edge.append("Low latency required (<50ms)")
    if scenario.get("privacy_sensitive", False):
        reasons_for_edge.append("Data privacy is critical")
    if scenario.get("offline_required", False):
        reasons_for_edge.append("Must work offline")
    if scenario.get("high_volume", False):
        reasons_for_edge.append("High volume (saves cloud costs)")

    if scenario.get("complex_model", False):
        reasons_for_cloud.append("Model too large for device")
    if scenario.get("needs_updates", False):
        reasons_for_cloud.append("Model needs frequent updates")
    if scenario.get("unlimited_compute", False):
        reasons_for_cloud.append("Task needs unlimited compute")

    edge_score = len(reasons_for_edge)
    cloud_score = len(reasons_for_cloud)
    recommendation = "EDGE" if edge_score > cloud_score else "CLOUD"

    return {
        "recommendation": recommendation,
        "edge_reasons": reasons_for_edge,
        "cloud_reasons": reasons_for_cloud,
    }


# Example scenarios
scenarios = [
    {
        "name": "Face unlock on phone",
        "latency_critical": True,
        "privacy_sensitive": True,
        "offline_required": True,
    },
    {
        "name": "Generating a 4K image from text",
        "complex_model": True,
        "unlimited_compute": True,
    },
    {
        "name": "Factory defect detection camera",
        "latency_critical": True,
        "high_volume": True,
        "offline_required": True,
    },
    {
        "name": "Chatbot with GPT-4 level quality",
        "complex_model": True,
        "needs_updates": True,
        "unlimited_compute": True,
    },
]

for scenario in scenarios:
    result = should_use_edge(scenario)
    print(f"\n{scenario['name']}")
    print(f"  Recommendation: {result['recommendation']}")
    if result['edge_reasons']:
        print(f"  Edge reasons: {', '.join(result['edge_reasons'])}")
    if result['cloud_reasons']:
        print(f"  Cloud reasons: {', '.join(result['cloud_reasons'])}")

## 6. Hands-On: Profile Your Own Model

Let's put it all together. We'll build a small model, profile it, and check if it's edge-ready.

In [ ]:
class MyEdgeModel(nn.Module):
    """A tiny model for CIFAR-10 using edge-friendly design."""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Regular conv for input
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
            # Depthwise separable block 1
            nn.Conv2d(16, 16, 3, padding=1, groups=16), nn.BatchNorm2d(16), nn.ReLU(),
            nn.Conv2d(16, 32, 1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            # Depthwise separable block 2
            nn.Conv2d(32, 32, 3, padding=1, groups=32), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(64, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


model = MyEdgeModel()

# Full profile
print("Model Profile:")
print("=" * 50)
model_info = summary(model, input_size=(1, 3, 32, 32), verbose=0)

params, size = get_model_size(model)
latency = benchmark_model(model, input_shape=(1, 3, 32, 32))

print(f"\nSummary:")
print(f"  Parameters:   {params:,}")
print(f"  Model size:   {size:.2f} MB")
print(f"  Latency:      {latency:.2f} ms")

# Edge-readiness check
print(f"\nEdge Readiness:")
print(f"  Phone (4GB RAM):          {'YES' if size < 100 else 'NO'}")
print(f"  Raspberry Pi (1GB RAM):   {'YES' if size < 50 else 'NO'}")
print(f"  Microcontroller (2MB):    {'YES' if size < 1 else 'NO'}")
print(f"  Real-time (< 33ms):       {'YES' if latency < 33 else 'NO'}")

## 7. Try It Yourself!

**Exercise 1:** Modify `MyEdgeModel` to use 3 depthwise separable blocks instead of 2. Does it get more accurate? How much bigger is it?

**Exercise 2:** Try profiling `models.squeezenet1_0()` — another classic edge model. How does it compare to MobileNetV3?

**Exercise 3:** Calculate: if you quantize a model from FP32 (4 bytes) to INT8 (1 byte), how much smaller does each model in our comparison table become?

## 8. Recap

**What we learned:**

- **Edge ML** = running AI on small devices (phones, cameras, sensors) instead of cloud servers
- **Why edge**: faster, more private, cheaper, works offline
- **The challenge**: limited memory and compute on small devices
- **Edge-friendly models** use tricks like depthwise separable convolutions to be small and fast
- **Profiling** (params, FLOPs, latency) tells you if a model fits on your target device
- **NPUs** are now standard in phones and laptops, making on-device AI increasingly practical

**What's next:** In Notebook 47, we'll learn three techniques to make **any** model smaller: **pruning**, **distillation**, and **quantization**.